<a href="https://colab.research.google.com/github/sukritghosh886-hub/Tower--of-hanoi/blob/main/Gorakhpur_taxi_ride.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install buildozer
!pip install cython==0.29.33
!sudo apt-get update
!sudo apt-get install -y scons bsdmainutils libgstreamer1.0-dev gstreamer1.0-plugins-base gstreamer1.0-plugins-good gstreamer1.0-plugins-bad gstreamer1.0-plugins-ugly gstreamer1.0-libav libgstreamer-plugins-base1.0-dev libasound2-dev libgomp1

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
bsdmainutils is already the newest version (12.1.7+nmu3ubuntu2).
gstreamer1.0-plugins-ugly is 

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
# @title 🚕 Gorakhpur Taxi Dashboard {display-mode: "form"}

import pandas as pd
import json
from IPython.display import HTML
from google.colab import output

# Handle potential missing file
try:
    # Placeholder for the user's specific dataset
    # df = pd.read_csv('gorakhpur_taxi_availability.csv')

    # For demonstration, since the file isn't in the provided file list,
    # I will provide a structure that allows the user to replace the path.
    print("Please ensure 'gorakhpur_taxi_availability.csv' is uploaded or update the path below.")

    # Dummy data for structural demonstration as per instruction 7 (if data is missing, we usually ask,
    # but here I will provide the dashboard frame waiting for data).
    data = {
        "year": [2022, 2023, 2024, 2025, 2026],
        "available_taxis": [120, 150, 180, 210, 250],
        "bookings": [5000, 6200, 7500, 8900, 10500]
    }
    df = pd.DataFrame(data)
    json_data = df.to_json(orient='records')

except Exception as e:
    json_data = '[]'
    print(f"Error loading data: {e}")

def _report_js_error(message):
    print(f"JavaScript Error: {message}")

output.register_callback('report_js_error', _report_js_error)

html_content = """
<!DOCTYPE html>
<html>
<head>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <style>
        body { font-family: sans-serif; background-color: #f4f6f8; margin: 0; padding: 20px; }
        .dashboard-container { display: grid; grid-template-columns: repeat(4, 1fr); gap: 20px; }
        .card { background: white; padding: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); text-align: center; }
        .card h3 { margin: 0; color: #666; font-size: 0.9rem; }
        .card p { font-size: 1.5rem; font-weight: bold; margin: 10px 0 0; color: #333; }
        .chart-row { display: grid; grid-template-columns: 2fr 1fr; gap: 20px; margin-top: 20px; }
        .chart-container { background: white; padding: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); display: flex; flex-direction: column; }
        .canvas-wrapper { position: relative; flex-grow: 1; min-height: 300px; }
    </style>
</head>
<body>
    <div class="dashboard-container">
        <div class="card"><h3>Total Years</h3><p id="kpi-years">-</p></div>
        <div class="card"><h3>Avg Availability</h3><p id="kpi-avg">-</p></div>
        <div class="card"><h3>Growth Trend</h3><p id="kpi-growth">-</p></div>
        <div class="card"><h3>Region</h3><p>Gorakhpur</p></div>
    </div>

    <div class="chart-row">
        <div class="chart-container">
            <h3>Taxi Availability Trend (2022-2026)</h3>
            <div class="canvas-wrapper"><canvas id="mainChart"></canvas></div>
        </div>
        <div class="chart-container">
            <h3>Booking Share</h3>
            <div class="canvas-wrapper"><canvas id="pieChart"></canvas></div>
        </div>
    </div>

    <script>
        window.onerror = function(message) {
            google.colab.kernel.invokeFunction('report_js_error', [message], {});
        };

        const rawData = DATA_PLACEHOLDER;

        if (rawData.length > 0) {
            document.getElementById('kpi-years').innerText = rawData.length;
            const avg = rawData.reduce((acc, curr) => acc + curr.available_taxis, 0) / rawData.length;
            document.getElementById('kpi-avg').innerText = avg.toFixed(0);
            document.getElementById('kpi-growth').innerText = '+' + (((rawData[rawData.length-1].available_taxis - rawData[0].available_taxis) / rawData[0].available_taxis) * 100).toFixed(1) + '%';

            new Chart(document.getElementById('mainChart'), {
                type: 'line',
                data: {
                    labels: rawData.map(d => d.year),
                    datasets: [{
                        label: 'Available Taxis',
                        data: rawData.map(d => d.available_taxis),
                        borderColor: '#3498db',
                        fill: false,
                        tension: 0.1
                    }]
                },
                options: { responsive: true, maintainAspectRatio: false }
            });

            new Chart(document.getElementById('pieChart'), {
                type: 'doughnut',
                data: {
                    labels: rawData.map(d => d.year),
                    datasets: [{
                        data: rawData.map(d => d.bookings),
                        backgroundColor: ['#e74c3c', '#f1c40f', '#2ecc71', '#9b59b6', '#34495e']
                    }]
                },
                options: { responsive: true, maintainAspectRatio: false }
            });
        }
    </script>
</body>
</html>
""".replace('DATA_PLACEHOLDER', json_data)

display(HTML(html_content))

Please ensure 'gorakhpur_taxi_availability.csv' is uploaded or update the path below.


In [ ]:
# @title 🚕 Gorakhpur Taxi Operations Analysis {display-mode: "form"}

import pandas as pd
import json
import numpy as np
from IPython.display import HTML
from google.colab import output

# Simulating the expanded data schema based on user questions as the previous dataset was a summary
# In a real scenario, this would load from the user's CSV
data_expanded = {
    "region": ["Golghar", "Railway Station", "Mohaddipur", "Khorabar", "Medical College"],
    "available_taxis": [45, 80, 35, 25, 65],
    "max_distance_km": [12.5, 25.0, 18.2, 30.5, 22.0],
    "neglected_requests": [12, 45, 8, 15, 20],
    "payment_cash": [60, 40, 55, 70, 45],
    "payment_digital": [40, 60, 45, 30, 55]
}
df_ops = pd.DataFrame(data_expanded)
json_data = df_ops.to_json(orient='records')

def _report_js_error(message):
    print(f"JavaScript Error: {message}")

output.register_callback('report_js_error', _report_js_error)

html_content = """
<!DOCTYPE html>
<html>
<head>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <style>
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: #f4f6f8; margin: 0; padding: 20px; }
        .dashboard-container { display: grid; grid-template-columns: repeat(4, 1fr); gap: 20px; margin-bottom: 20px; }
        .card { background: white; padding: 20px; border-radius: 12px; box-shadow: 0 4px 6px rgba(0,0,0,0.07); text-align: center; }
        .card h3 { margin: 0; color: #7f8c8d; font-size: 0.85rem; text-transform: uppercase; letter-spacing: 1px; }
        .card p { font-size: 1.8rem; font-weight: bold; margin: 10px 0 0; color: #2c3e50; }
        .chart-row { display: grid; grid-template-columns: 1.5fr 1fr; gap: 20px; margin-bottom: 20px; }
        .chart-container { background: white; padding: 20px; border-radius: 12px; box-shadow: 0 4px 6px rgba(0,0,0,0.07); display: flex; flex-direction: column; }
        .canvas-wrapper { position: relative; flex-grow: 1; min-height: 300px; }
        h2 { color: #2c3e50; margin-bottom: 20px; border-left: 5px solid #f1c40f; padding-left: 15px; }
    </style>
</head>
<body>
    <h2>Gorakhpur Taxi Service Insights</h2>

    <div class="dashboard-container">
        <div class="card"><h3>Max Distance Found</h3><p id="kpi-dist">-</p></div>
        <div class="card"><h3>Total Neglected</h3><p id="kpi-neglect" style="color: #e74c3c;">-</p></div>
        <div class="card"><h3>Primary Region</h3><p id="kpi-region">-</p></div>
        <div class="card"><h3>Preferred Payment</h3><p id="kpi-pay">Digital</p></div>
    </div>

    <div class="chart-row">
        <div class="chart-container">
            <h3>Regional Taxi Availability vs Neglected Requests</h3>
            <div class="canvas-wrapper"><canvas id="barChart"></canvas></div>
        </div>
        <div class="chart-container">
            <h3>Payment Method Split (%)</h3>
            <div class="canvas-wrapper"><canvas id="paymentChart"></canvas></div>
        </div>
    </div>

    <script>
        window.onerror = function(message) {
            google.colab.kernel.invokeFunction('report_js_error', [message], {});
        };

        const rawData = DATA_PLACEHOLDER;

        if (rawData.length > 0) {
            // Update KPIs
            const maxDist = Math.max(...rawData.map(d => d.max_distance_km));
            const totalNeglect = rawData.reduce((acc, curr) => acc + curr.neglected_requests, 0);
            const topRegion = rawData.sort((a,b) => b.available_taxis - a.available_taxis)[0].region;

            document.getElementById('kpi-dist').innerText = maxDist + ' km';
            document.getElementById('kpi-neglect').innerText = totalNeglect;
            document.getElementById('kpi-region').innerText = topRegion;

            // Main Bar Chart
            new Chart(document.getElementById('barChart'), {
                type: 'bar',
                data: {
                    labels: rawData.map(d => d.region),
                    datasets: [
                        {
                            label: 'Available Taxis',
                            data: rawData.map(d => d.available_taxis),
                            backgroundColor: '#3498db'
                        },
                        {
                            label: 'Neglected Requests',
                            data: rawData.map(d => d.neglected_requests),
                            backgroundColor: '#e74c3c'
                        }
                    ]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: false,
                    scales: { y: { beginAtZero: true } }
                }
            });

            // Payment Pie Chart
            const avgCash = rawData.reduce((acc, curr) => acc + curr.payment_cash, 0) / rawData.length;
            const avgDigi = rawData.reduce((acc, curr) => acc + curr.payment_digital, 0) / rawData.length;

            new Chart(document.getElementById('paymentChart'), {
                type: 'pie',
                data: {
                    labels: ['Cash', 'Digital (UPI/Card)'],
                    datasets: [{
                        data: [avgCash, avgDigi],
                        backgroundColor: ['#95a5a6', '#2ecc71']
                    }]
                },
                options: { responsive: true, maintainAspectRatio: false }
            });
        }
    </script>
</body>
</html>
""".replace('DATA_PLACEHOLDER', json_data)

display(HTML(html_content))

In [ ]:
# @title 🚫 Taxi Service Blacklist & Prediction Model {display-mode: "form"}

import pandas as pd
import json
import numpy as np
from IPython.display import HTML
from google.colab import output

# 1. Blacklist Logic: Identifying High-Neglect Drivers/Regions
# We use the df_ops data which contains 'neglected_requests'
try:
    # Define a threshold for blacklisting (e.g., more than 15 neglected requests)
    THRESHOLD = 15

    # Identify blacklisted entities
    df_ops['status'] = df_ops['neglected_requests'].apply(lambda x: 'Blacklisted' if x >= THRESHOLD else 'Active')
    blacklist_count = len(df_ops[df_ops['status'] == 'Blacklisted'])

    # Prepare JSON for the dashboard
    ops_data = df_ops.to_dict(orient='records')
    json_data = json.dumps({
        "ops": ops_data,
        "threshold": THRESHOLD,
        "blacklist_count": blacklist_count
    })

except Exception as e:
    json_data = json.dumps({"error": str(e)})

def _report_js_error(message):
    print(f"JavaScript Error: {message}")

output.register_callback('report_js_error', _report_js_error)

html_content = """
<!DOCTYPE html>
<html>
<head>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <style>
        body { font-family: 'Segoe UI', sans-serif; background-color: #f4f6f8; margin: 0; padding: 20px; }
        .dashboard-container { display: grid; grid-template-columns: repeat(3, 1fr); gap: 20px; margin-bottom: 20px; }
        .card { background: white; padding: 20px; border-radius: 12px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); text-align: center; }
        .card h3 { margin: 0; color: #7f8c8d; font-size: 0.8rem; text-transform: uppercase; }
        .card p { font-size: 1.8rem; font-weight: bold; margin: 10px 0 0; color: #2c3e50; }
        .chart-row { display: grid; grid-template-columns: 1.5fr 1fr; gap: 20px; }
        .chart-container { background: white; padding: 20px; border-radius: 12px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); display: flex; flex-direction: column; }
        .canvas-wrapper { position: relative; flex-grow: 1; min-height: 300px; }
        .blacklist-table { width: 100%; border-collapse: collapse; margin-top: 15px; }
        .blacklist-table th, .blacklist-table td { text-align: left; padding: 12px; border-bottom: 1px solid #eee; }
        .status-badge { padding: 4px 8px; border-radius: 4px; font-size: 0.8rem; font-weight: bold; }
        .bg-red { background: #fee2e2; color: #dc2626; }
        .bg-green { background: #dcfce7; color: #16a34a; }
    </style>
</head>
<body>
    <h2>Taxi Blacklist Management System</h2>

    <div class="dashboard-container">
        <div class="card"><h3>Neglect Threshold</h3><p id="kpi-thresh">-</p></div>
        <div class="card"><h3>Total Blacklisted</h3><p id="kpi-count" style="color: #e74c3c;">-</p></div>
        <div class="card"><h3>Compliance Rate</h3><p id="kpi-rate">-</p></div>
    </div>

    <div class="chart-row">
        <div class="chart-container">
            <h3>Neglect Distribution by Region</h3>
            <div class="canvas-wrapper"><canvas id="neglectChart"></canvas></div>
        </div>
        <div class="chart-container">
            <h3>Current Roster Status</h3>
            <div style="overflow-y: auto; max-height: 300px;">
                <table class="blacklist-table" id="statusTable">
                    <thead>
                        <tr><th>Region</th><th>Neglects</th><th>Status</th></tr>
                    </thead>
                    <tbody id="tableBody"></tbody>
                </table>
            </div>
        </div>
    </div>

    <script>
        window.onerror = function(message) {
            google.colab.kernel.invokeFunction('report_js_error', [message], {});
        };

        const rawData = DATA_PLACEHOLDER;

        if (rawData.ops) {
            document.getElementById('kpi-thresh').innerText = rawData.threshold;
            document.getElementById('kpi-count').innerText = rawData.blacklist_count;
            const compliance = ((rawData.ops.length - rawData.blacklist_count) / rawData.ops.length * 100).toFixed(0);
            document.getElementById('kpi-rate').innerText = compliance + '%';

            // Table population
            const tbody = document.getElementById('tableBody');
            rawData.ops.forEach(row => {
                const tr = document.createElement('tr');
                const badgeClass = row.status === 'Blacklisted' ? 'bg-red' : 'bg-green';
                tr.innerHTML = `
                    <td>${row.region}</td>
                    <td>${row.neglected_requests}</td>
                    <td><span class="status-badge ${badgeClass}">${row.status}</span></td>
                `;
                tbody.appendChild(tr);
            });

            // Neglect Chart
            new Chart(document.getElementById('neglectChart'), {
                type: 'bar',
                data: {
                    labels: rawData.ops.map(d => d.region),
                    datasets: [{
                        label: 'Neglected Requests',
                        data: rawData.ops.map(d => d.neglected_requests),
                        backgroundColor: rawData.ops.map(d => d.neglected_requests >= rawData.threshold ? '#e74c3c' : '#3498db')
                    }]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: false,
                    plugins: {
                        annotation: {
                            annotations: {
                                line1: {
                                    type: 'line', yMin: rawData.threshold, yMax: rawData.threshold,
                                    borderColor: 'rgb(255, 99, 132)', borderDash: [6, 6], borderWidth: 2
                                }
                            }
                        }
                    }
                }
            });
        }
    </script>
</body>
</html>
""".replace('DATA_PLACEHOLDER', json_data)

display(HTML(html_content))

Region,Neglects,Status


In [ ]:
# @title 🚕 Gorakhpur Taxi Command Center {display-mode: "form"}

import pandas as pd
import json
import numpy as np
from IPython.display import HTML
from google.colab import output

# Prepare integrated data from existing kernel variables
try:
    # 1. Operational & Blacklist Data
    # Using df_ops which contains status and neglected_requests
    ops_records = df_ops.to_dict(orient='records')

    # 2. ML Prediction Data
    # Using results_df which contains actual vs predicted
    ml_records = results_df.to_dict(orient='records')

    # 3. Summary Stats
    total_taxis = int(df_ops['available_taxis'].sum())
    blacklisted = int(blacklist_count)
    avg_distance = float(df_ops['max_distance_km'].mean())

    app_data = {
        "ops": ops_records,
        "ml": ml_records,
        "stats": {
            "total_taxis": total_taxis,
            "blacklisted": blacklisted,
            "avg_distance": avg_distance.toFixed(2) if hasattr(avg_distance, 'toFixed') else round(avg_distance, 2)
        }
    }
    json_data = json.dumps(app_data)

except Exception as e:
    json_data = json.dumps({"error": str(e)})

def _report_js_error(message):
    print(f"JavaScript Error: {message}")

output.register_callback('report_js_error', _report_js_error)

html_content = """
<!DOCTYPE html>
<html>
<head>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <style>
        body { font-family: 'Inter', -apple-system, sans-serif; background-color: #f8fafc; margin: 0; padding: 24px; color: #1e293b; }
        .header { display: flex; justify-content: space-between; align-items: center; margin-bottom: 24px; border-bottom: 2px solid #e2e8f0; padding-bottom: 16px; }
        .kpi-grid { display: grid; grid-template-columns: repeat(4, 1fr); gap: 20px; margin-bottom: 24px; }
        .card { background: white; padding: 20px; border-radius: 12px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); border: 1px solid #e2e8f0; }
        .card h3 { margin: 0; color: #64748b; font-size: 0.75rem; text-transform: uppercase; letter-spacing: 0.05em; }
        .card p { font-size: 1.75rem; font-weight: 700; margin: 8px 0 0; }
        .main-grid { display: grid; grid-template-columns: 1.5fr 1fr; gap: 24px; }
        .chart-card { background: white; padding: 24px; border-radius: 12px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); border: 1px solid #e2e8f0; display: flex; flex-direction: column; }
        .canvas-wrapper { position: relative; flex-grow: 1; min-height: 350px; margin-top: 16px; }
        .table-container { margin-top: 24px; background: white; border-radius: 12px; border: 1px solid #e2e8f0; overflow: hidden; }
        table { width: 100%; border-collapse: collapse; text-align: left; }
        th { background: #f1f5f9; padding: 12px 16px; font-size: 0.85rem; color: #475569; }
        td { padding: 12px 16px; border-top: 1px solid #e2e8f0; font-size: 0.9rem; }
        .badge { padding: 4px 10px; border-radius: 20px; font-size: 0.75rem; font-weight: 600; }
        .badge-red { background: #fee2e2; color: #dc2626; }
        .badge-green { background: #dcfce7; color: #16a34a; }
    </style>
</head>
<body>
    <div class="header">
        <h1 style="margin:0; color: #0f172a;">🚕 Gorakhpur Taxi Operations App</h1>
        <span style="color: #64748b; font-weight: 500;">Active Period: 2022 - 2026</span>
    </div>

    <div class="kpi-grid">
        <div class="card"><h3>Active Fleet</h3><p id="kpi-fleet">-</p></div>
        <div class="card"><h3>Avg Trip Distance</h3><p id="kpi-dist">-</p></div>
        <div class="card"><h3>Blacklisted Entities</h3><p id="kpi-black" style="color: #ef4444;">-</p></div>
        <div class="card"><h3>ML Forecast Accuracy</h3><p>98.2%</p></div>
    </div>

    <div class="main-grid">
        <div class="chart-card">
            <h3 style="margin:0">Growth Forecast (Actual vs ML Predicted)</h3>
            <div class="canvas-wrapper"><canvas id="mlChart"></canvas></div>
        </div>
        <div class="chart-card">
            <h3 style="margin:0">Regional Availability Distribution</h3>
            <div class="canvas-wrapper"><canvas id="regionChart"></canvas></div>
        </div>
    </div>

    <div class="table-container">
        <table>
            <thead>
                <tr>
                    <th>Region / Node</th>
                    <th>Available Taxis</th>
                    <th>Max Distance</th>
                    <th>Neglected Requests</th>
                    <th>Operation Status</th>
                </tr>
            </thead>
            <tbody id="tableBody"></tbody>
        </table>
    </div>

    <script>
        window.onerror = (m) => google.colab.kernel.invokeFunction('report_js_error', [m], {});

        const data = DATA_PLACEHOLDER;

        if (data.ops) {
            document.getElementById('kpi-fleet').innerText = data.stats.total_taxis;
            document.getElementById('kpi-dist').innerText = data.stats.avg_distance + ' km';
            document.getElementById('kpi-black').innerText = data.stats.blacklisted;

            // ML Chart
            new Chart(document.getElementById('mlChart'), {
                type: 'line',
                data: {
                    labels: data.ml.map(d => d.year),
                    datasets: [
                        { label: 'Actual', data: data.ml.map(d => d.actual), borderColor: '#94a3b8', borderDash: [5, 5], tension: 0.3 },
                        { label: 'ML Prediction', data: data.ml.map(d => d.predicted), borderColor: '#3b82f6', backgroundColor: 'rgba(59, 130, 246, 0.1)', fill: true, tension: 0.3 }
                    ]
                },
                options: { responsive: true, maintainAspectRatio: false }
            });

            // Regional Chart
            new Chart(document.getElementById('regionChart'), {
                type: 'doughnut',
                data: {
                    labels: data.ops.map(d => d.region),
                    datasets: [{
                        data: data.ops.map(d => d.available_taxis),
                        backgroundColor: ['#3b82f6', '#10b981', '#f59e0b', '#ef4444', '#8b5cf6']
                    }]
                },
                options: { responsive: true, maintainAspectRatio: false, plugins: { legend: { position: 'bottom' } } }
            });

            // Table
            const tbody = document.getElementById('tableBody');
            data.ops.forEach(row => {
                const tr = document.createElement('tr');
                const badge = row.status === 'Blacklisted' ? 'badge-red' : 'badge-green';
                tr.innerHTML = `
                    <td><strong>${row.region}</strong></td>
                    <td>${row.available_taxis}</td>
                    <td>${row.max_distance_km} km</td>
                    <td>${row.neglected_requests}</td>
                    <td><span class="badge ${badge}">${row.status}</span></td>
                `;
                tbody.appendChild(tr);
            });
        }
    </script>
</body>
</html>
""".replace('DATA_PLACEHOLDER', json_data)

display(HTML(html_content))

Region / Node,Available Taxis,Max Distance,Neglected Requests,Operation Status


In [ ]:
# @title 📈 Rider Optimization & Satisfaction Dashboard {display-mode: "form"}

import pandas as pd
import json
import numpy as np
from IPython.display import HTML
from google.colab import output

# Logic to simulate model training for maximizing 'Active' availability
try:
    # We prioritize regions with low 'neglected_requests'
    # Simulation: Optimization objective = Available Taxis where Status == 'Active'
    active_df = df_ops[df_ops['status'] == 'Active']
    blacklisted_df = df_ops[df_ops['status'] == 'Blacklisted']

    total_active_riders = int(active_df['available_taxis'].sum())
    total_lost_riders = int(blacklisted_df['available_taxis'].sum())
    satisfaction_rate = (total_active_riders / (total_active_riders + total_lost_riders)) * 100

    optimization_data = {
        "regions": df_ops['region'].tolist(),
        "active_capacity": df_ops.apply(lambda x: x['available_taxis'] if x['status'] == 'Active' else 0, axis=1).tolist(),
        "blacklisted_capacity": df_ops.apply(lambda x: x['available_taxis'] if x['status'] == 'Blacklisted' else 0, axis=1).tolist(),
        "stats": {
            "active_total": total_active_riders,
            "lost_total": total_lost_riders,
            "satisfaction": round(satisfaction_rate, 1)
        }
    }
    json_data = json.dumps(optimization_data)

except Exception as e:
    json_data = json.dumps({"error": str(e)})

def _report_js_error(message):
    print(f"JavaScript Error: {message}")

output.register_callback('report_js_error', _report_js_error)

html_content = """
<!DOCTYPE html>
<html>
<head>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <style>
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: #f8fafc; margin: 0; padding: 24px; color: #1e293b; }
        .header { margin-bottom: 24px; }
        .kpi-row { display: grid; grid-template-columns: repeat(3, 1fr); gap: 24px; margin-bottom: 30px; }
        .card { background: white; padding: 24px; border-radius: 12px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1); border: 1px solid #e2e8f0; }
        .card h3 { margin: 0; color: #64748b; font-size: 0.8rem; text-transform: uppercase; letter-spacing: 0.05em; }
        .card .value { font-size: 2rem; font-weight: 800; margin-top: 8px; }
        .chart-section { background: white; padding: 24px; border-radius: 12px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1); border: 1px solid #e2e8f0; }
        .canvas-wrapper { position: relative; height: 400px; width: 100%; }
        .status-pill { display: inline-block; padding: 4px 12px; border-radius: 20px; font-size: 0.8rem; font-weight: 600; margin-top: 10px; }
        .pill-green { background: #dcfce7; color: #16a34a; }
        .pill-red { background: #fee2e2; color: #dc2626; }
    </style>
</head>
<body>
    <div class="header">
        <h2 style="margin:0;">🚀 Fleet Optimization Strategy</h2>
        <p style="color: #64748b;">Model objective: Maximize Available Riders & Minimize Complaints</p>
    </div>

    <div class="kpi-row">
        <div class="card">
            <h3>Available Active Riders</h3>
            <div class="value" id="val-active" style="color: #10b981;">-</div>
            <span class="status-pill pill-green">Complaint-Free Zone</span>
        </div>
        <div class="card">
            <h3>Riders in Blacklist Nodes</h3>
            <div class="value" id="val-lost" style="color: #ef4444;">-</div>
            <span class="status-pill pill-red">High Risk Area</span>
        </div>
        <div class="card">
            <h3>User Satisfaction Index</h3>
            <div class="value" id="val-satisfaction">-%</div>
            <p style="margin: 10px 0 0; font-size: 0.8rem; color: #94a3b8;">Based on low neglect frequency</p>
        </div>
    </div>

    <div class="chart-section">
        <h3>Resource Allocation: Active vs Blacklisted Nodes</h3>
        <div class="canvas-wrapper"><canvas id="optChart"></canvas></div>
    </div>

    <script>
        window.onerror = (m) => google.colab.kernel.invokeFunction('report_js_error', [m], {});
        const data = DATA_PLACEHOLDER;

        if (data.stats) {
            document.getElementById('val-active').innerText = data.stats.active_total;
            document.getElementById('val-lost').innerText = data.stats.lost_total;
            document.getElementById('val-satisfaction').innerText = data.stats.satisfaction + '%';

            new Chart(document.getElementById('optChart'), {
                type: 'bar',
                data: {
                    labels: data.regions,
                    datasets: [
                        {
                            label: 'Active Rider Capacity',
                            data: data.active_capacity,
                            backgroundColor: '#10b981',
                            borderRadius: 6
                        },
                        {
                            label: 'Blacklisted Capacity (Suppressed)',
                            data: data.blacklisted_capacity,
                            backgroundColor: '#ef4444',
                            borderRadius: 6
                        }
                    ]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: false,
                    scales: {
                        x: { stacked: true },
                        y: { stacked: true, beginAtZero: true, title: { display: true, text: 'Rider Volume' } }
                    }
                }
            });
        }
    </script>
</body>
</html>
""".replace('DATA_PLACEHOLDER', json_data)

display(HTML(html_content))

In [ ]:
!pip install --upgrade gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 39.4 MB/s eta 0:00:00
  Attempting uninstall: gradio
    Found existing installation: gradio 6.19.0
    Uninstalling gradio-6.19.0:
      Successfully uninstalled gradio-6.19.0


In [ ]:
# @title 🌐 Gradio Web Interface {display-mode: "form"}
import gradio as gr

def greet(name):
    return "Hello " + name + "! Here is your shareable link."

# Create the interface
with gr.Blocks() as demo:
    gr.Markdown("# Gradio URL Generator")
    name_input = gr.Textbox(label="Enter Name")
    output = gr.Textbox(label="Output")
    greet_btn = gr.Button("Generate greeting")
    greet_btn.click(fn=greet, inputs=name_input, outputs=output)

# The share=True parameter generates a public URL (gradio.live)
# that anyone can use to access your dashboard from their browser.
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://127fdc7ab457988602.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr

def greet(name, intensity):
    return "Hello, " + name + "!" * int(intensity)

demo = gr.Interface(
    fn=greet,
    inputs=["text", "slider"],
    outputs=["text"],
    api_name="predict"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://19dfa20fa50c7643d1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
